
# GVH Diagonal Cubic 0.3.2.7.3.7.2.2 — Total EH + Directional Kinetic Hessian, Determinant Factorization and Degeneracy Surfaces

**Auteur :** Charlemagne O Laurince

Objectif :

\[
\boxed{Q_{\rm total}=Q_{\rm EH}+Q_u}
\]

puis audit du rang, de l'inversion générique et des surfaces de dégénérescence sur branches contrôlées.

\[
\boxed{\mathrm{DISPERSION\_READY=False}}
\]


In [1]:

import sympy as sp, json
from pathlib import Path
print("GVH 0.3.2.7.3.7.2.2")
print("SymPy:", sp.__version__)


GVH 0.3.2.7.3.7.2.2
SymPy: 1.14.0



## 1. Reconstruction du secteur directionnel de 7.2.1


In [2]:

c1,c2,c3,c4,kEH = sp.symbols("c1 c2 c3 c4 k_EH", real=True)
s = sp.symbols("s", real=True)
v1,v2,v3 = sp.symbols("v1 v2 v3", real=True)
K11,K22,K33,K12,K13,K23,S,W1,W2,W3 = sp.symbols(
    "K11 K22 K33 K12 K13 K23 S W1 W2 W3", real=True
)

K = sp.Matrix([[K11,K12,K13],[K12,K22,K23],[K13,K23,K33]])
v = sp.Matrix([v1,v2,v3])
W = sp.Matrix([W1,W2,W3])
vel = [K11,K22,K33,K12,K13,K23,S,W1,W2,W3]

A = -S
B = W-K*v
Cc = -K*v
D = s*K

def frob2(M):
    return sum(M[i,j]**2 for i in range(M.rows) for j in range(M.cols))

I1 = sp.expand(-A**2 + B.dot(B) - Cc.dot(Cc) + frob2(D))
theta = sp.expand(-A + sp.trace(D))
I3 = sp.expand(-A**2 + 2*B.dot(Cc) + sp.trace(D*D))
a_sp = s*B + D.T*v
a0 = v.dot(B)
a2 = sp.expand(-a0**2 + a_sp.dot(a_sp))

Lu = sp.expand(-c1*I1 - c2*theta**2 - c3*I3 + c4*a2)
Qu = sp.hessian(Lu, vel)
assert Qu.shape==(10,10)
assert Qu == Qu.T
print("Directional Hessian:", Qu.shape)


Directional Hessian: (10, 10)



## 2. Bloc cinétique Einstein-Hilbert

\[
\mathcal L_{\rm EH,kin}
=
\kappa_{\rm EH}(K_{ij}K^{ij}-K^2).
\]


In [3]:

KijKij = K11**2+K22**2+K33**2+2*K12**2+2*K13**2+2*K23**2
Ktr = K11+K22+K33
LEH = sp.expand(kEH*(KijKij-Ktr**2))
QEH = sp.hessian(LEH, vel)

print("EH Hessian rank in full 10D (kEH=1):", QEH.subs(kEH,1).rank())
assert QEH.shape==(10,10)


EH Hessian rank in full 10D (kEH=1): 6



## 3. Hessien total
\[
Q_{\rm total}=Q_{\rm EH}+Q_u.
\]


In [4]:

Qtot = Qu + QEH
assert Qtot.shape==(10,10)
assert Qtot == Qtot.T
print("Q_total:",Qtot.shape)


Q_total: (10, 10)



## 4. Témoin rationnel générique exact


In [5]:

generic_subs = {
    c1:sp.Rational(7,5), c2:sp.Rational(2,7),
    c3:sp.Rational(3,11), c4:sp.Rational(5,13),
    kEH:1, s:sp.Rational(6,5),
    v1:sp.Rational(1,5), v2:sp.Rational(1,7), v3:sp.Rational(1,11),
}
Qnum = sp.Matrix(Qtot.subs(generic_subs))
rank_generic = Qnum.rank()
det_generic = Qnum.det()

print("generic total rank =",rank_generic)
print("generic determinant nonzero =",det_generic != 0)
assert rank_generic==10
assert det_generic != 0


generic total rank = 10
generic determinant nonzero = True



## 5. Branche alignée \(v_i=0\)

Sur cette branche, le déterminant symbolique peut être factorisé de manière contrôlée.


In [6]:

Qv0 = sp.Matrix(Qtot.subs({v1:0,v2:0,v3:0}))
det_v0 = sp.factor(Qv0.det())
coeff_v0, factors_v0 = sp.factor_list(det_v0)

print("aligned determinant:")
sp.pprint(det_v0)
print("\nfactor families:")
for fac,pow_ in factors_v0:
    print("power",pow_,":",fac)


aligned determinant:
                  3                       5                                    ↪
     ⎛          2⎞  ⎛    2       2       ⎞  ⎛  2  2            2            2  ↪
8192⋅⎝-c₁ + c₄⋅s ⎠ ⋅⎝c₁⋅s  + c₃⋅s  - k_EH⎠ ⋅⎝c₁ ⋅s  + 2⋅c₁⋅c₂⋅s  + 2⋅c₁⋅c₃⋅s   ↪

↪                                                           
↪                        2                 2  2            ⎞
↪ + 2⋅c₁⋅k_EH + 2⋅c₂⋅c₃⋅s  - 2⋅c₂⋅k_EH + c₃ ⋅s  + 2⋅c₃⋅k_EH⎠

factor families:
power 3 : -c1 + c4*s**2
power 5 : c1*s**2 + c3*s**2 - k_EH
power 1 : c1**2*s**2 + 2*c1*c2*s**2 + 2*c1*c3*s**2 + 2*c1*k_EH + 2*c2*c3*s**2 - 2*c2*k_EH + c3**2*s**2 + 2*c3*k_EH



## 6. Contrôle de rang aligné


In [7]:

aligned_subs = {
    c1:2,c2:sp.Rational(1,3),c3:sp.Rational(2,5),c4:sp.Rational(1,7),
    kEH:1,s:1,v1:0,v2:0,v3:0
}
rank_aligned = Qtot.subs(aligned_subs).rank()
print("aligned generic rank =",rank_aligned)
assert rank_aligned==10


aligned generic rank = 10



## 7. Sous-blocs et complément de Schur


In [8]:

QKK = Qtot[:6,:6]
QKX = Qtot[:6,6:]
QXX = Qtot[6:,6:]

QKKnum = sp.Matrix(QKK.subs(generic_subs))
QKXnum = sp.Matrix(QKX.subs(generic_subs))
QXXnum = sp.Matrix(QXX.subs(generic_subs))

assert QKKnum.rank()==6
assert QXXnum.rank()==4
Schur = QXXnum - QKXnum.T*QKKnum.inv()*QKXnum
assert Schur.rank()==4
assert sp.simplify(QKKnum.det()*Schur.det()-Qnum.det())==0

print("QKK rank = 6")
print("QXX rank = 4")
print("Schur rank = 4")


QKK rank = 6
QXX rank = 4
Schur rank = 4



## 8. Inversion totale exacte sur témoin


In [9]:

Qinv = Qnum.inv()
assert Qnum*Qinv == sp.eye(10)
print("total exact-rational inverse: PASS")


total exact-rational inverse: PASS



## 9. Contrôle EH seul

Pour \(c_1=c_2=c_3=c_4=0\), le secteur directionnel ne fournit plus de cinétique aux quatre directions \((S,W_i)\). Le rang total doit donc chuter.


In [10]:

rank_EH_only = Qtot.subs({c1:0,c2:0,c3:0,c4:0,kEH:1}).rank()
print("EH-only rank in 10D =",rank_EH_only)
assert rank_EH_only < 10


EH-only rank in 10D = 6



## 10. Registre de dégénérescence

- branche générique non alignée : rang 10 démontré sur témoin rationnel exact ;
- branche alignée \(v_i=0\) : déterminant factorisé exactement ;
- secteur EH seul : chute de rang contrôlée ;
- factorisation symbolique générale non alignée : encore ouverte.


In [11]:

registry = {
    "generic_rank10_exists":True,
    "aligned_det_factorized":True,
    "aligned_factors":[(str(f),int(p)) for f,p in factors_v0],
    "EH_only_rank":int(rank_EH_only),
    "full_non_aligned_factorization":False,
}
registry


{'generic_rank10_exists': True,
 'aligned_det_factorized': True,
 'aligned_factors': [('-c1 + c4*s**2', 3),
  ('c1*s**2 + c3*s**2 - k_EH', 5),
  ('c1**2*s**2 + 2*c1*c2*s**2 + 2*c1*c3*s**2 + 2*c1*k_EH + 2*c2*c3*s**2 - 2*c2*k_EH + c3**2*s**2 + 2*c3*k_EH',
   1)],
 'EH_only_rank': 6,
 'full_non_aligned_factorization': False}


## 11. Verdict

Cette étape montre que la non-dégénérescence du Hessien directionnel **survit à l'ajout du bloc Einstein-Hilbert sur au moins une branche exacte** :

\[
\boxed{\operatorname{rank}Q_{\rm total}=10}.
\]

Mais cela ne ferme pas encore \(\mathcal C_\perp,\mathcal C_i\) ni l'algèbre hypersurface.


In [12]:

GATES = {
    "directional_Hessian_rebuilt":True,
    "EH_Hessian_added":True,
    "total_Hessian_constructed":True,
    "generic_total_rank10_exact_witness":True,
    "generic_total_inverse_exact_witness":True,
    "aligned_determinant_factorized":True,
    "EH_only_rank_drop_control":True,
    "full_non_aligned_det_factorization":False,
    "explicit_Cperp_total":False,
    "explicit_Ci_total":False,
    "hypersurface_algebra_closed":False,
}

CORE_PASS = all(list(GATES.values())[:7])
assert CORE_PASS
assert not all(GATES.values())

FINAL_STATUS = (
    "PARTIAL-PASS-TOTAL-EH-DIRECTIONAL-HESSIAN_"
    "GENERIC-RANK10-AND-ALIGNED-FACTORIZATION-PASS_"
    "BLOCKED-GENERAL-DEGENERACY-CLASSIFICATION-AND-CONSTRAINT-DENSITIES"
)
DISPERSION_READY=False

for k,vv in GATES.items():
    print(k,":",vv)
print("\nFINAL STATUS:",FINAL_STATUS)
print("DISPERSION_READY =",DISPERSION_READY)


directional_Hessian_rebuilt : True
EH_Hessian_added : True
total_Hessian_constructed : True
generic_total_rank10_exact_witness : True
generic_total_inverse_exact_witness : True
aligned_determinant_factorized : True
EH_only_rank_drop_control : True
full_non_aligned_det_factorization : False
explicit_Cperp_total : False
explicit_Ci_total : False
hypersurface_algebra_closed : False

FINAL STATUS: PARTIAL-PASS-TOTAL-EH-DIRECTIONAL-HESSIAN_GENERIC-RANK10-AND-ALIGNED-FACTORIZATION-PASS_BLOCKED-GENERAL-DEGENERACY-CLASSIFICATION-AND-CONSTRAINT-DENSITIES
DISPERSION_READY = False



## 12. Prochaine étape

### `0.3.2.7.3.7.2.3 — Exact Canonical Hamiltonian and Constraint Densities from the Total Kinetic Inverse`

Elle devra utiliser

\[
V=Q_{\rm total}^{-1}P
\]

dans la transformée de Legendre, puis dériver explicitement

\[
\mathcal C_\perp=\frac{\delta H_C}{\delta N},
\qquad
\mathcal C_i=\frac{\delta H_C}{\delta N^i}.
\]

\[
\boxed{\mathrm{DISPERSION\_READY=False}}
\]


In [13]:

artifact = {
    "notebook":"GVH_Diagonal_Cubic_0.3.2.7.3.7.2.2",
    "final_status":FINAL_STATUS,
    "generic_total_rank":int(rank_generic),
    "generic_total_det_nonzero":bool(det_generic != 0),
    "aligned_det_factorization":str(det_v0),
    "aligned_factor_list":[[str(f),int(p)] for f,p in factors_v0],
    "rank_aligned_generic":int(rank_aligned),
    "rank_EH_only":int(rank_EH_only),
    "gates":GATES,
    "dispersion_ready":False,
    "next":"GVH_Diagonal_Cubic_0.3.2.7.3.7.2.3_Exact_Canonical_Hamiltonian_and_Constraint_Densities_from_the_Total_Kinetic_Inverse.ipynb"
}

export_dir = Path("/content/gvh_exports") if Path("/content").exists() else Path.cwd()/"gvh_exports"
export_dir.mkdir(parents=True,exist_ok=True)
artifact_path = export_dir/"gvh_0.3.2.7.3.7.2.2_total_hessian_degeneracy.json"
artifact_path.write_text(json.dumps(artifact,indent=2),encoding="utf-8")
print("Artifact:",artifact_path)


Artifact: /content/gvh_exports/gvh_0.3.2.7.3.7.2.2_total_hessian_degeneracy.json



# Conclusion

\[
Q_{\rm total}=Q_{\rm EH}+Q_u
\]

est construit dans le même repère ADM local contrôlé que 7.2.1.

Un témoin rationnel exact donne

\[
\boxed{\operatorname{rank}Q_{\rm total}=10}
\]

et une inversion exacte réussie.

La branche alignée \(v_i=0\) fournit une factorisation exacte du déterminant, tandis que le contrôle EH seul produit bien une chute de rang dans l'espace cinétique total.

Verdict :

\[
\boxed{\text{PARTIAL PASS}}
\]

avec

\[
\boxed{\mathrm{DISPERSION\_READY=False}}.
\]
